# Part 2: Data Cleaning

In Part 1 I downloaded 107 raw JSON files (55 months of US imports + 52 months of India exports).
In this notebook I combine them into proper tables and fix the problems I found in the data.

**Problems I found (details below):**
1. Many columns are completely empty
2. Countries are numbers, not names
3. "World" total rows are mixed in with the country rows
4. Some quantities are 0 even though the phones were shipped
5. US quantities for 2022 are estimated, which gives fake prices
6. Very small shipments give silly prices (1 phone worth $9,500)
7. India's data for Jan to Mar 2022 is missing

In [1]:
import pandas as pd
import json
import glob

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

## Step 1: Load all the JSON files into one DataFrame

In [2]:
def load_folder(folder):
    all_rows = []
    for file in sorted(glob.glob(folder + "/*.json")):
        with open(file) as f:
            all_rows.extend(json.load(f)["data"])
    return pd.DataFrame(all_rows)

us_raw = load_folder("../data/raw/us_imports")
india_raw = load_folder("../data/raw/india_exports")

print("US imports:", us_raw.shape)
print("India exports:", india_raw.shape)

US imports: (2056, 47)
India exports: (2137, 47)


In [3]:
us_raw.head()

,typeCode,freqCode,refPeriodId,refYear,refMonth,period,reporterCode,reporterISO,reporterDesc,flowCode,flowDesc,partnerCode,partnerISO,partnerDesc,partner2Code,partner2ISO,partner2Desc,classificationCode,classificationSearchCode,isOriginalClassification,cmdCode,cmdDesc,aggrLevel,isLeaf,customsCode,customsDesc,mosCode,motCode,motDesc,qtyUnitCode,qtyUnitAbbr,qty,isQtyEstimated,altQtyUnitCode,altQtyUnitAbbr,altQty,isAltQtyEstimated,netWgt,isNetWgtEstimated,grossWgt,isGrossWgtEstimated,cifvalue,fobvalue,primaryValue,legacyEstimationFlag,isReported,isAggregate
0,C,M,20220101,2022,1,202201,842,None,None,M,None,0,None,None,0,None,None,H6,HS,True,851713,None,None,None,C00,None,0,0,None,5,None,2.356555e+07,True,-1,None,0.0,False,7437185.128,True,0.0,False,6.678247e+09,6.567945e+09,6.678247e+09,6,False,True
1,C,M,20220101,2022,1,202201,842,None,None,M,None,56,None,None,0,None,None,H6,HS,True,851713,None,None,None,C00,None,0,0,None,5,None,1.031650e+02,True,-1,None,0.0,False,32.558,True,0.0,False,2.923600e+04,2.917500e+04,2.923600e+04,6,True,False
2,C,M,20220101,2022,1,202201,842,None,None,M,None,76,None,None,0,None,None,H6,HS,True,851713,None,None,None,C00,None,0,0,None,5,None,4.834600e+02,True,-1,None,0.0,False,152.578,True,0.0,False,1.370080e+05,1.349330e+05,1.370080e+05,6,True,False
3,C,M,20220101,2022,1,202201,842,None,None,M,None,124,None,None,0,None,None,H6,HS,True,851713,None,None,None,C00,None,0,0,None,5,None,7.593140e+03,True,-1,None,0.0,False,2396.362,True,0.0,False,2.151822e+06,2.147320e+06,2.151822e+06,6,True,False
4,C,M,20220101,2022,1,202201,842,None,None,M,None,152,None,None,0,None,None,H6,HS,True,851713,None,None,None,C00,None,0,0,None,5,None,6.221560e+02,True,-1,None,0.0,False,196.350,True,0.0,False,1.763130e+05,1.763090e+05,1.763130e+05,6,True,False


In [4]:
us_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 2056 entries, 0 to 2055
Data columns (total 47 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   typeCode                  2056 non-null   str    
 1   freqCode                  2056 non-null   str    
 2   refPeriodId               2056 non-null   int64  
 3   refYear                   2056 non-null   int64  
 4   refMonth                  2056 non-null   int64  
 5   period                    2056 non-null   str    
 6   reporterCode              2056 non-null   int64  
 7   reporterISO               0 non-null      object 
 8   reporterDesc              0 non-null      object 
 9   flowCode                  2056 non-null   str    
 10  flowDesc                  0 non-null      object 
 11  partnerCode               2056 non-null   int64  
 12  partnerISO                0 non-null      object 
 13  partnerDesc               0 non-null      object 
 14  partner2Code       

## Step 2: Remove useless columns

47 columns is a lot. Many of them are completely empty because the free preview API doesn't fill in the
text descriptions (country names, product name, etc.).

In [5]:
empty_cols = [col for col in us_raw.columns if us_raw[col].isnull().all()]
print(len(empty_cols), "empty columns:")
print(empty_cols)

14 empty columns:
['reporterISO', 'reporterDesc', 'flowDesc', 'partnerISO', 'partnerDesc', 'partner2ISO', 'partner2Desc', 'cmdDesc', 'aggrLevel', 'isLeaf', 'customsDesc', 'motDesc', 'qtyUnitAbbr', 'altQtyUnitAbbr']


Some other columns have the **same value in every row**, so they don't tell us anything either:

In [6]:
same_value_cols = [col for col in us_raw.columns if us_raw[col].nunique() == 1]
for col in same_value_cols:
    print(f"{col:28} -> {us_raw[col].iloc[0]}")

typeCode                     -> C
freqCode                     -> M
reporterCode                 -> 842
flowCode                     -> M
partner2Code                 -> 0
classificationCode           -> H6
classificationSearchCode     -> HS
isOriginalClassification     -> True
cmdCode                      -> 851713
customsCode                  -> C00
mosCode                      -> 0
motCode                      -> 0
qtyUnitCode                  -> 5
isAltQtyEstimated            -> False
isNetWgtEstimated            -> True
grossWgt                     -> 0.0
isGrossWgtEstimated          -> False


This also confirms a few things: every row is **H6 (HS 2022)**, product **851713**, and `customsCode`, `motCode`
and `partner2Code` are always the "total" values (C00 / 0 / 0). So there are no breakdown rows by transport
mode that could be double counted.

I'll keep only the columns I actually need:

In [7]:
keep = ["period", "reporterCode", "flowCode", "partnerCode", "qty", "altQty", "qtyUnitCode",
        "isQtyEstimated", "netWgt", "cifvalue", "fobvalue", "primaryValue", "isAggregate"]

us = us_raw[keep].copy()
india = india_raw[keep].copy()
print(us.shape, india.shape)

(2056, 13) (2137, 13)


## Step 3: Convert country codes to country names

The data has `partnerCode = 699` instead of "India". I use the reference file from Part 1 to map them.

In [8]:
with open("../data/raw/partner_codes.json") as f:
    ref = json.load(f)["results"]

code_to_name = {row["id"]: row["text"] for row in ref}
print(len(code_to_name), "codes in reference file")

us["country"] = us["partnerCode"].map(code_to_name)
india["country"] = india["partnerCode"].map(code_to_name)

# check if any code didn't match
print("Unmatched in US data:", us["country"].isnull().sum())
print("Unmatched in India data:", india["country"].isnull().sum())

310 codes in reference file
Unmatched in US data: 0
Unmatched in India data: 0


All codes matched, but some UN names are long or unusual. The important one is **"Other Asia, nes"**: this is how
the UN lists **Taiwan** in trade data (code 490), so I renamed it. "Areas, nes" means the destination was not
specified.

In [9]:
rename_countries = {
    "Other Asia, nes": "Taiwan",
    "Viet Nam": "Vietnam",
    "USA": "United States",
    "China, Hong Kong SAR": "Hong Kong",
    "China, Macao SAR": "Macao",
    "Rep. of Korea": "South Korea",
    "Russian Federation": "Russia",
    "United Rep. of Tanzania": "Tanzania",
    "Rep. of Moldova": "Moldova",
    "Dem. Rep. of the Congo": "DR Congo",
    "Dominican Rep.": "Dominican Republic",
    "Bosnia Herzegovina": "Bosnia and Herzegovina",
    "Br. Virgin Isds": "British Virgin Islands",
    "Areas, nes": "Unspecified",
}
us["country"] = us["country"].replace(rename_countries).str.strip()
india["country"] = india["country"].replace(rename_countries).str.strip()

us["country"].value_counts().head(10)

country
World          55
Canada         55
China          55
Germany        55
Hong Kong      55
Japan          55
South Korea    55
Mexico         55
Taiwan         55
India          55
Name: count, dtype: int64

## Step 4: Separate the "World" rows

Every month has one extra row where the partner is **"World"** (code 0). It's the total of all countries.
If I leave it in, every sum would be doubled.

At first I tried to remove these rows with the `isAggregate` column, because the name sounds like "total row".
But look at what happens with India's data:

In [10]:
print("US   - isAggregate=True rows:", us["isAggregate"].sum(), "  World rows:", (us["partnerCode"] == 0).sum())
print("India- isAggregate=True rows:", india["isAggregate"].sum(), "  World rows:", (india["partnerCode"] == 0).sum())

US   - isAggregate=True rows: 55   World rows: 55
India- isAggregate=True rows: 1695   World rows: 52


For India, `isAggregate` is True for 1,695 rows, not just the 52 World rows. So what does it actually mean?

In [11]:
print("US:    isReported is always the opposite of isAggregate ->", (us_raw["isReported"] == ~us_raw["isAggregate"]).all())
print("India: isReported is always the opposite of isAggregate ->", (india_raw["isReported"] == ~india_raw["isAggregate"]).all())

US:    isReported is always the opposite of isAggregate -> True
India: isReported is always the opposite of isAggregate -> True


`isAggregate` is just the opposite of `isReported`. It means **Comtrade calculated this number itself** (for India,
by adding up India's more detailed 8-digit codes) instead of the country reporting it directly. It has nothing to do
with the World total. Using it would have deleted most of India's data, so **I used `partnerCode == 0` instead.**

Before removing the World rows, I checked that the countries add up to the World total for every month:

In [12]:
def check_world_total(df, name):
    world = df[df["partnerCode"] == 0].set_index("period")["primaryValue"]
    countries = df[df["partnerCode"] != 0].groupby("period")["primaryValue"].sum()
    difference = (world - countries).abs()
    print(f"{name}: {len(world)} months checked, biggest difference = ${difference.max():.2f}")

check_world_total(us, "US imports")
check_world_total(india, "India exports")

US imports: 55 months checked, biggest difference = $0.00
India exports: 52 months checked, biggest difference = $0.01


The countries add up to the World total (the tiny India difference is just rounding). So now I can safely
move the World rows into their own table.

In [13]:
us_world = us[us["partnerCode"] == 0].copy()
india_world = india[india["partnerCode"] == 0].copy()

us = us[us["partnerCode"] != 0].copy()
india = india[india["partnerCode"] != 0].copy()

print("US country rows:", len(us), " | India country rows:", len(india))

US country rows: 2001  | India country rows: 2085


## Step 5: Fix quantities that are 0

Some rows say `qty = 0`, which doesn't make sense when millions of dollars of phones were shipped. But the
`altQty` column (alternative quantity) has the number:

In [14]:
india_all = pd.concat([india, india_world])
india_all[india_all["qty"] == 0][["period", "country", "qty", "altQty", "qtyUnitCode", "primaryValue"]]

,period,country,qty,altQty,qtyUnitCode,primaryValue
785,202310,Tuvalu,0.0,0.0,5,7.056220e+02
1006,202403,Tuvalu,0.0,0.0,5,1.612033e+03
1554,202504,Ghana,0.0,2500.0,-1,1.237480e+02
1587,202505,DR Congo,0.0,1995.0,-1,1.587000e+01
1862,202512,Vietnam,0.0,34059.0,-1,3.034688e+03
1543,202504,World,0.0,7116234.0,-1,2.325552e+09
1579,202505,World,0.0,8615961.0,-1,2.963212e+09
1838,202512,World,0.0,6664730.0,-1,2.506895e+09


For example, India's **World** total for April 2025 shows 0 phones but $2.3 billion in value, while `altQty`
says 7.1 million phones. In most of these rows `qtyUnitCode` is -1 (unit missing), but `altQty` is in unit 5
("number of items"). So I copy `altQty` into `qty` where `qty` is 0. The two Tuvalu rows have 0 in both,
so their quantity is unknown.

In [15]:
for df in [us, india, us_world, india_world]:
    fix = df["qty"] == 0
    df.loc[fix, "qty"] = df.loc[fix, "altQty"]
    df.loc[df["qty"] == 0, "qty"] = None   # still 0 = unknown

print("India rows with unknown quantity:", india["qty"].isnull().sum())
print("US rows with unknown quantity:", us["qty"].isnull().sum())

India rows with unknown quantity: 2
US rows with unknown quantity: 0


## Step 6: Dates and values

`period` is text like "202506". I turn it into a real date and add year / month columns.

For the value I use `primaryValue`, the main value Comtrade uses. For imports it is the **CIF** value (includes
shipping and insurance to the US) and for exports it is the **FOB** value (price when it leaves India).
That difference matters later when I compare the two countries' numbers.

In [16]:
def add_columns(df, flow_name):
    df["date"] = pd.to_datetime(df["period"], format="%Y%m")
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["month_name"] = df["date"].dt.strftime("%b")
    df["flow"] = flow_name
    df["value_usd"] = df["primaryValue"]
    df["value_usd_mn"] = (df["primaryValue"] / 1_000_000).round(3)
    df["net_weight_kg"] = df["netWgt"]
    return df

us = add_columns(us, "US import")
india = add_columns(india, "India export")
us_world = add_columns(us_world, "US import")
india_world = add_columns(india_world, "India export")

us[["period", "date", "year", "month", "country", "value_usd_mn"]].head()

,period,date,year,month,country,value_usd_mn
1,202201,2022-01-01,2022,1,Belgium,0.029
2,202201,2022-01-01,2022,1,Brazil,0.137
3,202201,2022-01-01,2022,1,Canada,2.152
4,202201,2022-01-01,2022,1,Chile,0.176
5,202201,2022-01-01,2022,1,China,4962.411


## Step 7: Price per phone and the 2022 problem

Price per phone = value / number of phones. This is useful because it tells us *what kind* of phones each country
makes (cheap or premium). But when I calculated it by year, 2022 looked strange:

In [17]:
us["price_per_phone"] = us["value_usd"] / us["qty"]

main3 = us[us["country"].isin(["India", "China", "Vietnam"])]
check = main3.groupby(["year", "country"]).apply(
    lambda g: g["value_usd"].sum() / g["qty"].sum(), include_groups=False).unstack().round(0)
check

country,China,India,Vietnam
year,,,
2022,283.0,283.0,283.0
2023,377.0,456.0,433.0
2024,355.0,466.0,303.0
2025,389.0,432.0,238.0
2026,357.0,533.0,242.0


In 2022 all three countries have **exactly $283** per phone. That is impossible in real life. The reason is in the
`isQtyEstimated` column:

In [18]:
us.groupby("year")["isQtyEstimated"].mean().map("{:.0%}".format)

year
2022    100%
2023      0%
2024      0%
2025      0%
2026      0%
Name: isQtyEstimated, dtype: str

**100% of the 2022 US quantities are estimated** by Comtrade. The US didn't report phone counts for 2022, so
Comtrade guessed them from the dollar value using one average price. The 2022 **values are real**, but 2022
**quantities and prices are not**. I keep the rows (the values are fine) but set the 2022 price to empty,
so I don't use fake prices by mistake.

In [19]:
us.loc[us["isQtyEstimated"], "price_per_phone"] = None
us = us.rename(columns={"isQtyEstimated": "qty_estimated"})
india = india.rename(columns={"isQtyEstimated": "qty_estimated"})

## Step 8: Small shipments give silly prices

Sorting by price shows some strange rows:

In [20]:
us.dropna(subset=["price_per_phone"]).sort_values("price_per_phone")[
    ["period", "country", "qty", "value_usd", "price_per_phone"]].iloc[[0, 1, 2, -3, -2, -1]]

,period,country,qty,value_usd,price_per_phone
1370,202410,South Africa,527.0,4194.0,7.958254
1890,202602,Mexico,414.0,3315.0,8.007246
1992,202605,South Africa,217.0,2170.0,10.000000
912,202310,Uruguay,1.0,9169.0,9169.000000
2033,202607,Croatia,1.0,9504.0,9504.000000
1487,202502,Austria,1.0,9525.0,9525.000000


A single "phone" from Croatia for $9,504, or 527 phones from South Africa at $8 each. These are tiny shipments
(samples, repairs, returns or wrong codes). They are real rows, so I keep them in the totals, but they would ruin the
average price. How much do they matter?

In [21]:
small = us["qty"] < 1000
print("Rows with less than 1,000 phones:", small.sum(), "out of", len(us))
print("Their share of total value: {:.3%}".format(us.loc[small, "value_usd"].sum() / us["value_usd"].sum()))

Rows with less than 1,000 phones: 1223 out of 2001
Their share of total value: 0.028%


More than half the rows, but only **0.03% of the value**. So for price per phone I only use rows with at least 1,000
phones.

In [22]:
us.loc[us["qty"] < 1000, "price_per_phone"] = None
india["price_per_phone"] = india["value_usd"] / india["qty"]
india.loc[(india["qty"] < 1000) | india["qty"].isnull(), "price_per_phone"] = None

us["price_per_phone"] = us["price_per_phone"].round(2)
india["price_per_phone"] = india["price_per_phone"].round(2)

us["price_per_phone"].describe()

count     600.000000
mean      250.737083
std       162.015727
min        10.400000
25%       105.680000
50%       229.755000
75%       370.582500
max      1130.030000
Name: price_per_phone, dtype: float64

## Step 9: Missing months

Checking which months each dataset has:

In [23]:
all_months = pd.date_range("2022-01-01", "2026-07-01", freq="MS")
us_missing = [d.strftime("%b %Y") for d in all_months if d not in set(us["date"])]
india_missing = [d.strftime("%b %Y") for d in all_months if d not in set(india["date"])]
print("US missing months:", us_missing)
print("India missing months:", india_missing)

US missing months: []
India missing months: ['Jan 2022', 'Feb 2022', 'Mar 2022']


India has no smartphone data for **Jan to Mar 2022**. I checked the API again with the older code **851712** and
found those three months there, under the old HS 2017 system (classification "H5"). In Comtrade, India's data only
moves to the new HS 2022 codes from **April 2022**, the start of India's financial year. Code 851712 includes all mobile phones (not just smartphones),
so I can't mix it in. **India's analysis starts from April 2022.**

## Step 10: Final checks and save

In [24]:
final_cols = ["date", "year", "month", "month_name", "flow", "partnerCode", "country",
              "value_usd", "value_usd_mn", "qty", "qty_estimated", "price_per_phone", "net_weight_kg"]

us_clean = us[final_cols].rename(columns={"partnerCode": "country_code", "qty": "quantity"})
india_clean = india[final_cols].rename(columns={"partnerCode": "country_code", "qty": "quantity"})

us_clean = us_clean.sort_values(["date", "value_usd"], ascending=[True, False]).reset_index(drop=True)
india_clean = india_clean.sort_values(["date", "value_usd"], ascending=[True, False]).reset_index(drop=True)

# duplicates check: one row per month per country
print("US duplicates:", us_clean.duplicated(subset=["date", "country"]).sum())
print("India duplicates:", india_clean.duplicated(subset=["date", "country"]).sum())
print("Negative values:", (us_clean["value_usd"] < 0).sum() + (india_clean["value_usd"] < 0).sum())

us_clean.head()

US duplicates: 0
India duplicates: 0
Negative values: 0


,date,year,month,month_name,flow,country_code,country,value_usd,value_usd_mn,quantity,qty_estimated,price_per_phone,net_weight_kg
0,2022-01-01,2022,1,Jan,US import,156,China,4.962411e+09,4962.411,1.751087e+07,True,NaN,5526355.620
1,2022-01-01,2022,1,Jan,US import,704,Vietnam,1.531798e+09,1531.798,5.405258e+06,True,NaN,1705876.003
2,2022-01-01,2022,1,Jan,US import,410,South Korea,1.203038e+08,120.304,4.245165e+05,True,NaN,133975.558
3,2022-01-01,2022,1,Jan,US import,699,India,2.892808e+07,28.928,1.020786e+05,True,NaN,32215.564
4,2022-01-01,2022,1,Jan,US import,344,Hong Kong,2.145917e+07,21.459,7.572304e+04,True,NaN,23897.864


In [25]:
world_totals = pd.concat([us_world, india_world])[["date", "year", "month", "flow", "value_usd", "value_usd_mn", "qty"]]
world_totals = world_totals.rename(columns={"qty": "quantity"}).sort_values(["flow", "date"]).reset_index(drop=True)

us_clean.to_csv("../data/cleaned/us_smartphone_imports.csv", index=False)
india_clean.to_csv("../data/cleaned/india_smartphone_exports.csv", index=False)
world_totals.to_csv("../data/cleaned/monthly_world_totals.csv", index=False)

with pd.ExcelWriter("../data/cleaned/smartphone_trade_cleaned.xlsx") as writer:
    us_clean.to_excel(writer, sheet_name="US Imports", index=False)
    india_clean.to_excel(writer, sheet_name="India Exports", index=False)
    world_totals.to_excel(writer, sheet_name="Monthly Totals", index=False)

print("Saved 3 CSV files + 1 Excel file")

Saved 3 CSV files + 1 Excel file


## Cleaning summary

In [26]:
summary = pd.DataFrame({
    "Dataset": ["US imports", "India exports"],
    "Raw rows": [len(us_raw), len(india_raw)],
    "World rows moved out": [len(us_world), len(india_world)],
    "Clean rows": [len(us_clean), len(india_clean)],
    "Countries": [us_clean["country"].nunique(), india_clean["country"].nunique()],
    "Months": [us_clean["date"].nunique(), india_clean["date"].nunique()],
    "From": [us_clean["date"].min().strftime("%b %Y"), india_clean["date"].min().strftime("%b %Y")],
    "To": [us_clean["date"].max().strftime("%b %Y"), india_clean["date"].max().strftime("%b %Y")],
})
summary

,Dataset,Raw rows,World rows moved out,Clean rows,Countries,Months,From,To
0,US imports,2056,55,2001,109,55,Jan 2022,Jul 2026
1,India exports,2137,52,2085,108,52,Apr 2022,Jul 2026


Columns went from 47 to 13, every country has a readable name, the totals are checked against the World
rows, and the fake 2022 prices and the tiny-shipment prices are blanked out.

Next: `03_analysis.ipynb`